# CHISCo Closed-Set EEG Classification Baseline

This notebook implements the **task-optimized closed-set classifier** used as the conventional reference system in *Semantic Alignment of EEG and Text: A Contrastive Learning Framework for Decoding Imagined Speech*.

The model predicts one of **39 semantic categories** from imagined-speech EEG recordings in the **Chinese Imagined Speech Corpus (CHISCo)**. It uses an EEGNet-inspired convolutional feature extractor, sinusoidal positional encoding, a Transformer encoder block, and a 39-way softmax classification head.

Training is performed **within subject** using sentence-grouped five-fold cross-validation, with a separate validation partition for checkpoint selection. Performance is evaluated using **Top-k accuracy, Macro-F1, and balanced 2v2 accuracy**, together with prior-aware random baselines for the classification metrics.

Unlike the proposed contrastive framework in the paper, this baseline does **not** use a text encoder or map EEG into a language embedding space; it is optimized directly for closed-set category classification.


## Reproducible Setup and Experiment Configuration

This cell initializes the software environment and defines the experiment-wide configuration used throughout the notebook. It sets deterministic random seeds, TensorFlow/GPU behavior, subject-specific data paths, output directories, cross-validation settings, training hyperparameters, evaluation parameters, and plotting utilities.

The default configuration runs all five folds for `sub-01`, but the subject and data/output locations can be changed through the corresponding variables or environment paths before execution.


In [ ]:
import os

GLOBAL_SEED = 1337
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import gc
import hashlib
import json
import math
import pickle
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from tensorflow.keras import Model, layers
from tqdm.auto import tqdm

SEED = GLOBAL_SEED

def derive_seed(*parts, base_seed=GLOBAL_SEED):
    payload = "|".join(map(str, (base_seed, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little") & 0x7FFFFFFF

def seed_everything(seed=GLOBAL_SEED):
    random.seed(int(seed))
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))

seed_everything()
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Could not enable TensorFlow op determinism:", exc)
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as exc:
    print("Could not set GPU memory growth:", exc)
tf.config.optimizer.set_jit(False)

SUBJECT_ID = "sub-01"
DATA_ROOTS = {
    "sub-01": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub01/chisco_sub01_ready"),
    "sub-02": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub02/Chisco-IS-sub02"),
    "sub-03": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub03/Chisco-IS-sub03"),
    "sub-04": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub04/Chisco-IS-sub04"),
    "sub-05": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub05/Chisco-IS-sub05"),
}
DATA_ROOT = Path(os.getenv("CHISCO_DATA_ROOT", DATA_ROOTS[SUBJECT_ID]))
WORK_DIR = Path(os.getenv("CHISCO_OUTPUT_DIR", f"/kaggle/working/chisco_classifier_{SUBJECT_ID}"))
TEMP_DIR = Path(os.getenv("CHISCO_TEMP_DIR", f"/kaggle/temp/chisco_classifier_{SUBJECT_ID}"))
PLOTS_DIR, CACHE_DIR, CKPT_DIR = WORK_DIR / "plots", TEMP_DIR / "cache", TEMP_DIR / "checkpoints"
for path in (PLOTS_DIR, CACHE_DIR, CKPT_DIR):
    path.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
VAL_RATIO_OF_TRAIN = 0.15
FOLDS_TO_RUN = list(range(N_FOLDS))
RUN_TRAINING = True
NUM_CLASSES, TARGET_C, TARGET_T = 39, 122, 1651
DROP_LAST_N_CHANNELS = 3
PER_TRIAL_ZSCORE = True
PKL_FILE_KEYWORD = "imagine"
TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 16, 128
MAX_EPOCHS, LEARNING_RATE = 100, 5e-4
EARLY_STOPPING_PATIENCE, MIN_DELTA = 9, 1e-3
REDUCE_LR_PATIENCE, REDUCE_LR_FACTOR, MIN_LR = 4, 0.7, 1e-6
LABEL_SMOOTHING = 0.0
TOPK_LIST = [1, 2, 3, 5, 10]
N_RANDOM_BASELINE, N_2V2_PAIRS = 10000, 20000
EEG_CACHE_DTYPE = np.float16
FORCE_REBUILD_EEG_CACHE = False
LOAD_EEG_TO_RAM = True

PAPER_PALETTE = {"model": "#C46A3A", "random": "#A9A9A9"}

def save_pdf(filename):
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.show()
    plt.close()

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Subject:", SUBJECT_ID)
print("Data root:", DATA_ROOT)
print("PDF output:", PLOTS_DIR)

## Label Mapping, Dataset Indexing, and EEG Preprocessing

This cell defines the **39 CHISCo semantic categories**, maps the original Chinese category names to English labels for readability, and links each stimulus sentence to its class. It then scans the preprocessed imagined-speech files and builds a trial-level index containing the sentence, semantic label, source file, and sample position for every usable EEG trial.

Each EEG trial is converted to a fixed **122 × 1651** representation. The final three non-EEG channels are removed, optional per-trial Z-score normalization is applied, and signals are center-cropped or zero-padded when necessary to obtain a consistent temporal length.


In [ ]:
chinese_to_english = {
    "预订和旅行安排": "Travel Arrangements",
    "住房和设施": "Housing and Facilities",
    "自然和天气": "Nature and Weather",
    "个人行为和日常活动": "Personal Behavior and Daily Activities",
    "金融和付款": "Finance",
    "饮食和用餐": "Food and Dining",
    "旅行和行李管理": "Travel Affairs",
    "交通和出行": "Transportation and Commuting",
    "旅游和度假": "Vacation",
    "设备故障和环境问题": "Equipment Malfunction or Environmental Issues",
    "价格和费用": "Prices and Costs",
    "时间和日程安排": "Time and Scheduling",
    "衣物和服饰": "Clothing",
    "语言和学习": "Learning",
    "欢迎和感谢": "Welcoming and Thanking",
    "道歉和请求原谅": "Apologies",
    "个人信息": "Personal Information",
    "询问和个人事务": "Inquiries and Personal Matters",
    "饮食习惯": "Eating Habits",
    "产品和质量保证": "Products and Quality Assurance",
    "健康和安全建议": "Health and Safety",
    "家庭关系和家庭事件": "Family Relationships and Events",
    "提供和请求帮助": "Providing and Requesting Assistance",
    "理发和美容护理": "Hairdressing and Beauty Care",
    "节日和庆祝活动": "Festivals and Celebrations",
    "互联网和信息技术": "Internet and Information Technology",
    "健康和身体不适": "Physical Discomfort",
    "情感和人际关系": "Emotions and Interpersonal Relationships",
    "社交和聚会活动": "Gathering Activities",
    "人际交往和情感表达": "Social Interactions",
    "问候和情感状态": "Greetings",
    "工作和职场交流": "Work",
    "表演艺术": "Performing Arts",
    "教育和学习": "Education",
    "娱乐和媒体消费": "Entertainment and Media Consumption",
    "求职和职业发展": "Job Hunting and Career Development",
    "健身": "Fitness",
    "兴趣爱好": "Hobbies",
    "体育和运动": "Sports",
}
CLASS_PHRASES_ZH = list(chinese_to_english)
CLASS_PHRASES_EN = [chinese_to_english[label] for label in CLASS_PHRASES_ZH]
CLASS2ID = {label: i for i, label in enumerate(CLASS_PHRASES_ZH)}
assert len(CLASS2ID) == NUM_CLASSES

def find_dataset_root(root):
    root = Path(root)
    if (root / "textdataset").exists() and (root / "derivatives" / "preprocessed_pkl").exists():
        return root
    candidates = [p.parent for p in root.rglob("textdataset") if (p.parent / "derivatives" / "preprocessed_pkl").exists()]
    if not candidates:
        raise FileNotFoundError(f"Could not find the CHISCo dataset under {root}")
    return candidates[0]

def load_textdataset_mapping(text_dir):
    frames = [pd.read_excel(path) for path in sorted(Path(text_dir).glob("*.xlsx"))]
    frames = [frame for frame in frames if frame.shape[1] >= 2]
    if not frames:
        raise RuntimeError(f"No valid textdataset Excel files found in {text_dir}")
    data = pd.concat(frames, ignore_index=True)
    columns = [str(column) for column in data.columns]
    def find_column(candidates, default_index):
        return next((column for candidate in candidates for column in columns if candidate.lower() in column.lower()), columns[default_index])
    sentence_col = find_column(["句子", "sentence", "text"], 0)
    label_col = find_column(["标签", "label", "class"], 1)
    mapping = dict(zip(data[sentence_col].astype(str).str.strip(), data[label_col].astype(str).str.strip()))
    mapping = {sentence: label for sentence, label in mapping.items() if label in CLASS2ID}
    if not mapping:
        raise RuntimeError("No known class labels were found in the textdataset")
    return mapping

def load_pickle_list(path):
    with open(path, "rb") as file:
        data = pickle.load(file)
    if not isinstance(data, list):
        raise TypeError(f"Expected a list in {path}, got {type(data)}")
    return data

def to_channels_time(sample_input):
    x = np.asarray(sample_input)
    if x.ndim == 3 and x.shape[0] == 1:
        x = np.squeeze(x, axis=0)
    if x.ndim == 3 and x.shape[-1] == 1:
        x = np.squeeze(x, axis=-1)
    if x.ndim != 2:
        raise ValueError(f"Unexpected input shape: {x.shape}")
    if x.shape[0] > 300 and x.shape[1] <= 512:
        x = x.T
    return x.astype(np.float32, copy=False)

def center_crop_or_pad(x, target_t=TARGET_T):
    _, t = x.shape
    if t > target_t:
        start = (t - target_t) // 2
        return x[:, start:start + target_t]
    if t < target_t:
        left = (target_t - t) // 2
        return np.pad(x, ((0, 0), (left, target_t - t - left)), mode="constant")
    return x

def preprocess_eeg(sample_input):
    """Convert one trial to a normalized 122 × 1651 EEG array."""
    x = to_channels_time(sample_input)
    if x.shape[0] >= TARGET_C + DROP_LAST_N_CHANNELS:
        x = x[:-DROP_LAST_N_CHANNELS]
    else:
        keep_c = min(TARGET_C, x.shape[0])
        x = x[:keep_c]
        if keep_c != TARGET_C:
            x = np.pad(x, ((0, TARGET_C - keep_c), (0, 0)), mode="constant")
    if PER_TRIAL_ZSCORE:
        x = (x - x.mean(keepdims=True)) / (x.std(keepdims=True) + 1e-6)
    return np.nan_to_num(center_crop_or_pad(x)).astype(np.float32)

def collect_trial_metadata(eeg_paths):
    rows, skipped_text, skipped_sample = [], 0, 0
    for path in tqdm(eeg_paths, desc="scan trial metadata"):
        for sample_idx, sample in enumerate(load_pickle_list(path)):
            sentence = str(sample.get("text", "")).strip()
            label = TEXT2CLASS.get(sentence)
            if label not in CLASS2ID:
                skipped_text += 1
                continue
            if "input_features" not in sample:
                skipped_sample += 1
                continue
            rows.append({
                "trial_uid": len(rows), "pkl_path": str(path), "pkl_file": path.name,
                "sample_idx": int(sample_idx), "sentence_zh": sentence, "label_zh": label,
                "label_en": chinese_to_english[label], "label_id": int(CLASS2ID[label]),
            })
    if not rows:
        raise RuntimeError("No valid imagined-speech trials were collected")
    print(f"Kept {len(rows)} trials; skipped {skipped_text} unknown texts and {skipped_sample} invalid samples")
    return pd.DataFrame(rows)

def trial_index_signature(index_df):
    columns = ["trial_uid", "pkl_file", "sample_idx", "sentence_zh", "label_id"]
    hasher = hashlib.sha256()
    for row in index_df[columns].itertuples(index=False, name=None):
        hasher.update(("\t".join(map(str, row)) + "\n").encode("utf-8"))
    return hasher.hexdigest()

DATA_ROOT = find_dataset_root(DATA_ROOT)
TEXTDATASET_DIR = DATA_ROOT / "textdataset"
PKL_DIR = DATA_ROOT / "derivatives" / "preprocessed_pkl"
EEG_PATHS = sorted(path for path in PKL_DIR.rglob("*.pkl") if PKL_FILE_KEYWORD.lower() in path.name.lower())
if not EEG_PATHS:
    raise RuntimeError(f"No imagined-speech pickle files found under {PKL_DIR}")
TEXT2CLASS = load_textdataset_mapping(TEXTDATASET_DIR)
trial_index = collect_trial_metadata(EEG_PATHS)
y_class = trial_index["label_id"].to_numpy(np.int64)
display(trial_index.head(3))
print("Classes present:", trial_index["label_id"].nunique())

## Build and Load the Preprocessed EEG Cache

To avoid repeatedly loading and preprocessing the original pickle files during cross-validation, this cell stores the indexed EEG trials in a reusable **memory-mapped cache**. Cache metadata records the subject, tensor shape, preprocessing settings, data type, and a signature of the trial index so that an incompatible or outdated cache is automatically rebuilt.

The resulting array preserves the trial order defined above and has shape **(trials, 122 channels, 1651 samples)**. Depending on the configuration, it can remain memory-mapped or be loaded into RAM for faster model training.


In [ ]:
def expected_memmap_bytes(shape, dtype):
    return int(np.prod(shape) * np.dtype(dtype).itemsize)

def build_eeg_cache(index_df):
    shape = (len(index_df), TARGET_C, TARGET_T)
    tag = "trialzscore" if PER_TRIAL_ZSCORE else "rawvolts"
    cache_path = CACHE_DIR / f"{SUBJECT_ID}_imagined_{tag}_center_n{shape[0]}_{TARGET_C}x{TARGET_T}_{np.dtype(EEG_CACHE_DTYPE).name}.dat"
    meta_path = cache_path.with_suffix(".json")
    metadata = {
        "schema_version": 2,
        "subject_id": SUBJECT_ID,
        "shape": list(shape),
        "dtype": np.dtype(EEG_CACHE_DTYPE).name,
        "target_c": TARGET_C,
        "target_t": TARGET_T,
        "per_trial_zscore": PER_TRIAL_ZSCORE,
        "volts_to_microvolts_scale": False,
        "crop": "center",
        "drop_last_n_channels": DROP_LAST_N_CHANNELS,
        "trial_index_signature_sha256": trial_index_signature(index_df),
    }
    valid_cache = False
    if cache_path.exists() and meta_path.exists() and not FORCE_REBUILD_EEG_CACHE:
        with open(meta_path, "r", encoding="utf-8") as file:
            saved_metadata = json.load(file)
        valid_cache = cache_path.stat().st_size == expected_memmap_bytes(shape, EEG_CACHE_DTYPE) and saved_metadata == metadata
    if valid_cache:
        print("Using EEG cache:", cache_path)
    else:
        cache_path.unlink(missing_ok=True)
        meta_path.unlink(missing_ok=True)
        print("Building EEG cache:", cache_path)
        cache = np.memmap(cache_path, dtype=EEG_CACHE_DTYPE, mode="w+", shape=shape)
        for pkl_path, group in tqdm(index_df.groupby("pkl_path", sort=False), desc="preprocess EEG"):
            samples = load_pickle_list(Path(pkl_path))
            for row in group.itertuples(index=False):
                cache[int(row.trial_uid)] = preprocess_eeg(samples[int(row.sample_idx)] ["input_features"]).astype(EEG_CACHE_DTYPE)
            cache.flush()
            del samples
            gc.collect()
        with open(meta_path, "w", encoding="utf-8") as file:
            json.dump(metadata, file, indent=2, ensure_ascii=False)
        del cache
        gc.collect()
    return np.memmap(cache_path, dtype=EEG_CACHE_DTYPE, mode="r", shape=shape), cache_path

X_EEG_MEMMAP, EEG_CACHE_PATH = build_eeg_cache(trial_index)
X_EEG = np.asarray(X_EEG_MEMMAP, dtype=np.float16) if LOAD_EEG_TO_RAM else X_EEG_MEMMAP
first_sample = np.asarray(X_EEG_MEMMAP[0], dtype=np.float32)
print("EEG source:", type(X_EEG).__name__, X_EEG.shape, X_EEG.dtype)
print("First sample mean/std:", float(first_sample.mean()), float(first_sample.std()))

## Sentence-Grouped Five-Fold Cross-Validation

This cell constructs the within-subject cross-validation scheme used for model development and evaluation. Trials are first grouped by their **original stimulus sentence**, ensuring that repeated trials of the same sentence cannot appear in different train, validation, or test partitions.

A stratified five-fold split is created at the sentence-group level. In each outer fold, one partition is held out for testing, while **15% of the remaining sentence groups** are reserved for validation. Additional checks verify that the partitions are disjoint, contain no sentence leakage, and that every trial appears in exactly one outer test fold.


In [ ]:
def make_sentence_groups(index_df):
    group_df = index_df.groupby("sentence_zh", sort=True).agg(
        label_id=("label_id", "first"), label_en=("label_en", "first"), n_trials=("trial_uid", "size")
    ).reset_index()
    if (index_df.groupby("sentence_zh")["label_id"].nunique() > 1).any():
        raise ValueError("At least one sentence maps to multiple labels")
    group_df["group_id"] = np.arange(len(group_df), dtype=np.int64)
    sentence_to_group = dict(zip(group_df["sentence_zh"], group_df["group_id"]))
    trial_group_ids = index_df["sentence_zh"].map(sentence_to_group).to_numpy(np.int64)
    return group_df, trial_group_ids

def indices_for_groups(trial_group_ids, selected_groups):
    return np.flatnonzero(np.isin(trial_group_ids, np.asarray(selected_groups, dtype=np.int64))).astype(np.int64)

def build_grouped_cv_splits(index_df):
    group_df, trial_group_ids = make_sentence_groups(index_df)
    group_ids = group_df["group_id"].to_numpy(np.int64)
    group_labels = group_df["label_id"].to_numpy(np.int64)
    if np.bincount(group_labels, minlength=NUM_CLASSES).min() < N_FOLDS:
        raise ValueError(f"Each class needs at least {N_FOLDS} sentence groups")
    outer = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=GLOBAL_SEED)
    splits = []
    for fold_id, (trainval_pos, test_pos) in enumerate(outer.split(group_ids, group_labels)):
        trainval_groups, test_groups = group_ids[trainval_pos], group_ids[test_pos]
        trainval_labels = group_labels[trainval_pos]
        inner = StratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=GLOBAL_SEED)
        train_rel, val_rel = next(inner.split(trainval_groups, trainval_labels))
        train_groups, val_groups = trainval_groups[train_rel], trainval_groups[val_rel]
        splits.append({
            "fold_id": fold_id,
            "train_idx": indices_for_groups(trial_group_ids, train_groups),
            "val_idx": indices_for_groups(trial_group_ids, val_groups),
            "test_idx": indices_for_groups(trial_group_ids, test_groups),
            "train_group_ids": train_groups.astype(np.int64),
            "val_group_ids": val_groups.astype(np.int64),
            "test_group_ids": test_groups.astype(np.int64),
        })
    return splits, group_df

def validate_grouped_cv_splits(index_df, splits):
    all_trials = np.arange(len(index_df), dtype=np.int64)
    all_test = []
    if len(splits) != N_FOLDS:
        raise AssertionError(f"Expected {N_FOLDS} folds, found {len(splits)}")
    for split in splits:
        train, val, test = split["train_idx"], split["val_idx"], split["test_idx"]
        if any(np.intersect1d(a, b).size for a, b in ((train, val), (train, test), (val, test))):
            raise AssertionError(f"Fold {split['fold_id']} contains overlapping trial indices")
        if not np.array_equal(np.sort(np.concatenate([train, val, test])), all_trials):
            raise AssertionError(f"Fold {split['fold_id']} does not cover every trial once")
        sentence_sets = [set(index_df.iloc[idx]["sentence_zh"]) for idx in (train, val, test)]
        if sentence_sets[0] & sentence_sets[1] or sentence_sets[0] & sentence_sets[2] or sentence_sets[1] & sentence_sets[2]:
            raise AssertionError(f"Fold {split['fold_id']} contains sentence leakage")
        all_test.append(test)
    if not np.array_equal(np.sort(np.concatenate(all_test)), all_trials):
        raise AssertionError("Each trial must appear in exactly one outer test fold")

def summarize_splits(splits, group_df):
    group_labels = group_df["label_id"].to_numpy(np.int64)
    rows = []
    for split in splits:
        for name in ("train", "val", "test"):
            idx = split[f"{name}_idx"]
            groups = split[f"{name}_group_ids"]
            trial_counts = np.bincount(y_class[idx], minlength=NUM_CLASSES)
            group_counts = np.bincount(group_labels[groups], minlength=NUM_CLASSES)
            rows.append({
                "fold": split["fold_id"], "split": name, "trials": len(idx), "sentence_groups": len(groups),
                "classes": len(np.unique(y_class[idx])), "trial_count_min": trial_counts.min(),
                "trial_count_max": trial_counts.max(), "group_count_min": group_counts.min(),
                "group_count_max": group_counts.max(),
            })
    return pd.DataFrame(rows)

CV_SPLITS, sentence_group_df = build_grouped_cv_splits(trial_index)
validate_grouped_cv_splits(trial_index, CV_SPLITS)
split_summary_df = summarize_splits(CV_SPLITS, sentence_group_df)
display(split_summary_df)
print("Sentence groups:", len(sentence_group_df), "| Trials:", len(trial_index))

## Closed-Set EEGNet-Transformer Classifier

This cell defines the neural architecture used for the closed-set baseline. An EEGNet-inspired convolutional backbone performs temporal and spatial feature extraction, after which sinusoidal positional encoding and a Transformer block model the resulting EEG feature sequence.

The encoded sequence is summarized with global average pooling and passed to a **39-class softmax output layer**. The cell also defines normalized inverse-square-root class weights, mini-batch generation, and the weighted categorical cross-entropy objective used to account for class imbalance.


In [ ]:
def chisco_eeg_feature_extractor(inputs, dropout_rate=0.1):
    x = layers.Conv2D(8, (1, 20), padding="same", use_bias=False, name="layer1_conv")(inputs)
    x = layers.BatchNormalization(axis=-1, momentum=0.9, epsilon=1e-5, name="layer1_bn")(x)
    x = layers.DepthwiseConv2D((TARGET_C, 1), use_bias=False, depth_multiplier=8, name="layer1_depthwise")(x)
    x = layers.BatchNormalization(axis=-1, momentum=0.9, epsilon=1e-5, name="layer1_depthwise_bn")(x)
    x = layers.Activation("elu", name="layer1_elu")(x)
    x = layers.AveragePooling2D((1, 2), name="layer1_pool")(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.SeparableConv2D(64, (1, 10), use_bias=False, padding="same", name="layer2_sep")(x)
    x = layers.BatchNormalization(axis=-1, momentum=0.9, epsilon=1e-5, name="layer2_bn")(x)
    x = layers.Activation("elu", name="layer2_elu")(x)
    x = layers.AveragePooling2D((1, 5), name="layer2_pool")(x)
    x = layers.Dropout(dropout_rate)(x)
    return layers.Lambda(lambda tensor: tf.squeeze(tensor, axis=1), name="squeeze_channels")(x)

class PositionalEncoding(layers.Layer):
    """Sinusoidal encoding for the EEG token sequence."""
    def call(self, x):
        seq_len, d_model = tf.shape(x)[1], tf.shape(x)[2]
        position = tf.cast(tf.range(seq_len)[:, None], tf.float32)
        div_term = tf.exp(tf.range(0, d_model, 2, dtype=tf.float32) * -(math.log(10000.0) / tf.cast(d_model, tf.float32)))
        encoding = tf.concat([tf.sin(position * div_term), tf.cos(position * div_term)], axis=-1)[:, :d_model]
        return x + encoding[None]

def transformer_block(x, num_heads=8, d_model=64, ff_dim=256, dropout=0.1):
    attention = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, output_shape=d_model, dropout=dropout
    )(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention)
    feed_forward = layers.Dense(ff_dim, activation="relu")(x)
    feed_forward = layers.Dropout(dropout)(feed_forward)
    feed_forward = layers.Dense(d_model)(feed_forward)
    return layers.LayerNormalization(epsilon=1e-6)(x + feed_forward)

def build_eeg_classifier(input_shape=(TARGET_C, TARGET_T, 1), num_classes=NUM_CLASSES, cnn_dropout=0.5, trans_dropout=0.1):
    inputs = layers.Input(shape=input_shape, name="eeg_input")
    x = chisco_eeg_feature_extractor(inputs, dropout_rate=cnn_dropout)
    x = PositionalEncoding()(x)
    x = layers.Dropout(trans_dropout)(x)
    x = transformer_block(x, num_heads=8, d_model=64, ff_dim=256, dropout=trans_dropout)
    x = layers.Dropout(trans_dropout)(x)
    x = layers.GlobalAveragePooling1D(name="mean_pooling")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    return Model(inputs, outputs, name="Chisco_EEGNet_Transformer_Classifier")

def calculate_class_weights(labels, num_classes=NUM_CLASSES):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    frequency = (counts + 1e-6) / (counts.sum() + 1e-6)
    weights = (1.0 / np.sqrt(frequency))
    return tf.constant((weights / weights.mean()).astype(np.float32)), counts.astype(int)

def iter_batches(indices, labels, batch_size=64, training=False, drop_remainder=False, shuffle_seed=None):
    indices = np.asarray(indices, dtype=np.int64)
    if training:
        if shuffle_seed is None:
            raise ValueError("shuffle_seed is required for training batches")
        order = np.random.default_rng(int(shuffle_seed)).permutation(indices)
    else:
        order = indices.copy()
    if drop_remainder:
        order = order[:(len(order) // batch_size) * batch_size]
    for start in range(0, len(order), batch_size):
        batch_idx = order[start:start + batch_size]
        yield X_EEG[batch_idx].astype(np.float32, copy=False)[..., None], labels[batch_idx].astype(np.int64, copy=False)

@tf.function
def weighted_classification_loss(probs, labels, class_weights):
    one_hot = tf.one_hot(labels, depth=NUM_CLASSES)
    losses = tf.keras.losses.categorical_crossentropy(
        one_hot, probs, from_logits=False, label_smoothing=LABEL_SMOOTHING
    )
    return tf.reduce_mean(losses * tf.gather(class_weights, labels))

classifier_preview = build_eeg_classifier()
classifier_preview.summary()
del classifier_preview
gc.collect()

## Fold-Wise Training and Checkpoint Selection

This cell trains a separate classifier for each cross-validation fold using only that fold's training partition. Optimization uses Adam with fold-specific class weights, learning-rate reduction based on validation loss, and early stopping to limit unnecessary training once validation performance stops improving.

For the closed-set system, the saved checkpoint is selected by **validation Top-1 accuracy**; validation loss is used as a tie-breaker when Top-1 values are equal. Training histories are plotted for each fold, and the notebook can either train new models or load previously saved fold checkpoints depending on the configuration.


In [ ]:
def plot_training_history(history, fold_id, best_epoch):
    history_df = pd.DataFrame(history)
    _, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(history_df["epoch"], history_df["loss"], label="Train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation")
    axes[0].axvline(best_epoch, linestyle=":", linewidth=1.6, label=f"Best val Top-1 (epoch {best_epoch})")
    axes[0].set(xlabel="Epoch", ylabel="Weighted cross-entropy", title=f"Fold {fold_id}: loss")
    axes[1].plot(history_df["epoch"], history_df["top1"], label="Train Top-1")
    axes[1].plot(history_df["epoch"], history_df["val_top1"], label="Validation Top-1")
    axes[1].axvline(best_epoch, linestyle=":", linewidth=1.6, label=f"Best val Top-1 (epoch {best_epoch})")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy", title=f"Fold {fold_id}: Top-1 accuracy")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend(frameon=False)
    save_pdf(f"{SUBJECT_ID}_fold{fold_id}_training_history")

def train_one_fold(split):
    fold_id = int(split["fold_id"])
    train_idx = np.asarray(split["train_idx"], dtype=np.int64)
    val_idx = np.asarray(split["val_idx"], dtype=np.int64)
    fold_seed = derive_seed("model_and_tf", fold_id)
    seed_everything(fold_seed)
    run_dir = CKPT_DIR / f"{SUBJECT_ID}_fold{fold_id}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "best_val_top1.weights.h5"
    class_weights, class_counts = calculate_class_weights(y_class[train_idx])
    print("=" * 80)
    print(f"Fold {fold_id} | train {len(train_idx)} | val {len(val_idx)} | test {len(split['test_idx'])}")
    print("Class count min/max:", class_counts.min(), class_counts.max())

    tf.keras.backend.clear_session()
    gc.collect()
    seed_everything(fold_seed)
    classifier = build_eeg_classifier(cnn_dropout=0.5, trans_dropout=0.1)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    train_loss, val_loss = tf.keras.metrics.Mean(), tf.keras.metrics.Mean()
    train_top1, val_top1 = tf.keras.metrics.SparseCategoricalAccuracy(), tf.keras.metrics.SparseCategoricalAccuracy()

    @tf.function(reduce_retracing=True)
    def train_step(x, labels):
        with tf.GradientTape() as tape:
            probs = classifier(x, training=True)
            loss = weighted_classification_loss(probs, labels, class_weights)
        gradients = tape.gradient(loss, classifier.trainable_variables)
        optimizer.apply_gradients([(g, v) for g, v in zip(gradients, classifier.trainable_variables) if g is not None])
        train_loss.update_state(loss)
        train_top1.update_state(labels, probs)

    @tf.function(reduce_retracing=True)
    def validation_step(x, labels):
        probs = classifier(x, training=False)
        loss = weighted_classification_loss(probs, labels, class_weights)
        val_loss.update_state(loss)
        val_top1.update_state(labels, probs)

    best_loss, best_top1, best_top1_loss = np.inf, -np.inf, np.inf
    best_epoch = None
    bad_epochs_es = bad_epochs_lr = 0
    best_lr_loss = np.inf
    history = []
    train_steps = len(train_idx) // TRAIN_BATCH_SIZE
    val_steps = int(np.ceil(len(val_idx) / TRAIN_BATCH_SIZE))

    for epoch in range(1, MAX_EPOCHS + 1):
        start_time = time.time()
        for metric in (train_loss, train_top1, val_loss, val_top1):
            metric.reset_state()
        shuffle_seed = derive_seed("batch_order", fold_id, epoch)
        train_iter = iter_batches(train_idx, y_class, TRAIN_BATCH_SIZE, True, True, shuffle_seed)
        for x_batch, labels in tqdm(train_iter, total=train_steps, desc=f"fold {fold_id} ep {epoch:03d} train", leave=False):
            train_step(x_batch, labels)
        val_iter = iter_batches(val_idx, y_class, TRAIN_BATCH_SIZE)
        for x_batch, labels in tqdm(val_iter, total=val_steps, desc=f"fold {fold_id} ep {epoch:03d} val", leave=False):
            validation_step(x_batch, labels)

        row = {
            "epoch": epoch, "loss": float(train_loss.result().numpy()), "top1": float(train_top1.result().numpy()),
            "val_loss": float(val_loss.result().numpy()), "val_top1": float(val_top1.result().numpy()),
            "learning_rate": float(optimizer.learning_rate.numpy()), "seconds": time.time() - start_time,
        }
        if not np.isfinite(row["val_loss"]):
            raise FloatingPointError(f"Fold {fold_id}, epoch {epoch}: non-finite validation loss")
        notes = []
        if row["val_loss"] < best_loss - MIN_DELTA:
            best_loss = row["val_loss"]
            bad_epochs_es = bad_epochs_lr = 0
            best_lr_loss = row["val_loss"]
        else:
            bad_epochs_es += 1
            if row["val_loss"] < best_lr_loss - MIN_DELTA:
                best_lr_loss = row["val_loss"]
                bad_epochs_lr = 0
            else:
                bad_epochs_lr += 1

        top1_improved = row["val_top1"] > best_top1 + 1e-12
        tied_with_lower_loss = np.isclose(row["val_top1"], best_top1, atol=1e-12, rtol=0.0) and row["val_loss"] < best_top1_loss
        if top1_improved or tied_with_lower_loss:
            best_top1, best_top1_loss, best_epoch = row["val_top1"], row["val_loss"], epoch
            classifier.save_weights(checkpoint_path)
            notes.append("saved best val Top-1")
        if bad_epochs_lr >= REDUCE_LR_PATIENCE:
            old_lr = float(optimizer.learning_rate.numpy())
            new_lr = max(old_lr * REDUCE_LR_FACTOR, MIN_LR)
            optimizer.learning_rate.assign(new_lr)
            bad_epochs_lr = 0
            notes.append(f"lr {old_lr:.2e}->{new_lr:.2e}")
        history.append(row)
        print(
            f"Fold {fold_id} Ep {epoch:03d} | loss {row['loss']:.4f} val {row['val_loss']:.4f} | "
            f"top1 {row['top1']:.3f}/{row['val_top1']:.3f} | {row['seconds']:.1f}s"
            + (" | " + "; ".join(notes) if notes else "")
        )
        if bad_epochs_es >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping fold {fold_id} at epoch {epoch}")
            break

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No checkpoint was created for fold {fold_id}")
    plot_training_history(history, fold_id, best_epoch)
    del classifier
    gc.collect()
    return {
        "fold_id": fold_id, "fold_seed": fold_seed, "checkpoint": str(checkpoint_path),
        "best_epoch": int(best_epoch), "best_val_top1": float(best_top1), "tie_break_val_loss": float(best_top1_loss),
        "train_idx": train_idx.tolist(), "val_idx": val_idx.tolist(),
        "test_idx": np.asarray(split["test_idx"], dtype=np.int64).tolist(),
    }

run_records = []
for split in CV_SPLITS:
    fold_id = int(split["fold_id"])
    if fold_id not in FOLDS_TO_RUN:
        continue
    if RUN_TRAINING:
        run_records.append(train_one_fold(split))
    else:
        run_records.append({
            "fold_id": fold_id, "fold_seed": derive_seed("model_and_tf", fold_id),
            "checkpoint": str(CKPT_DIR / f"{SUBJECT_ID}_fold{fold_id}" / "best_val_top1.weights.h5"),
            "train_idx": np.asarray(split["train_idx"], dtype=np.int64).tolist(),
            "val_idx": np.asarray(split["val_idx"], dtype=np.int64).tolist(),
            "test_idx": np.asarray(split["test_idx"], dtype=np.int64).tolist(),
        })

run_summary_df = pd.DataFrame([{k: v for k, v in record.items() if not k.endswith("_idx")} for record in run_records])
display(run_summary_df)

## Held-Out Evaluation, Random Baselines, and Balanced 2v2

This cell reloads the selected checkpoint from each fold and computes class probabilities for the corresponding **held-out test trials**. From these predictions, it calculates Top-k accuracy, Macro-F1, and the probability-based **balanced 2v2 accuracy** used for the closed-set classifier.

For Top-k accuracy and Macro-F1, prior-aware random performance is estimated by Monte Carlo sampling from the class distribution of the fold-specific training set. Balanced 2v2 is estimated from class-balanced pairs of test trials, with a theoretical chance level of **0.50**. The resulting metrics are reported separately for each fold.


In [ ]:
def predict_indices(classifier, indices, batch_size=EVAL_BATCH_SIZE):
    probabilities, labels = [], []
    iterator = iter_batches(indices, y_class, batch_size=batch_size)
    for x_batch, y_batch in tqdm(iterator, total=int(np.ceil(len(indices) / batch_size)), leave=False):
        probabilities.append(classifier(x_batch, training=False).numpy().astype(np.float32))
        labels.append(y_batch.astype(np.int64))
    return np.concatenate(probabilities), np.concatenate(labels)

def build_prediction_cache(records):
    items = []
    for record in records:
        checkpoint = Path(record["checkpoint"])
        if not checkpoint.exists():
            raise FileNotFoundError(f"Missing checkpoint: {checkpoint}")
        tf.keras.backend.clear_session()
        seed_everything(record["fold_seed"])
        classifier = build_eeg_classifier(cnn_dropout=0.5, trans_dropout=0.1)
        classifier.load_weights(checkpoint)
        test_idx = np.asarray(record["test_idx"], dtype=np.int64)
        print(f"Predicting fold {record['fold_id']} from {checkpoint.name}")
        probs, labels = predict_indices(classifier, test_idx)
        items.append({
            "fold_id": int(record["fold_id"]), "test_idx": test_idx, "y_true": labels,
            "probs": probs, "ranked": np.argsort(-probs, axis=1),
            "train_idx": np.asarray(record["train_idx"], dtype=np.int64),
        })
        del classifier
        gc.collect()
    return items

def mean_std(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0

def topk_accuracy(labels, ranked, k):
    return float((ranked[:, :k] == np.asarray(labels)[:, None]).any(axis=1).mean())

def automatic_run_chunk(n_trials, n_classes, target_mb=80, min_chunk=20, max_chunk=500):
    bytes_per_run = n_trials * n_classes * np.dtype(np.float32).itemsize
    return int(np.clip((target_mb * 1024**2) / max(bytes_per_run, 1), min_chunk, max_chunk))

def prior_aware_random_topk(labels, class_prior, ks=TOPK_LIST, n_runs=N_RANDOM_BASELINE, seed=SEED, desc="random baseline"):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=np.int64)
    class_prior = np.asarray(class_prior, dtype=np.float32)
    class_prior /= class_prior.sum()
    ks, max_k = sorted(ks), max(ks)
    chunk_size = automatic_run_chunk(len(labels), len(class_prior))
    log_prior = np.log(class_prior + 1e-12).astype(np.float32)
    sums, sums_squared, total = {k: 0.0 for k in ks}, {k: 0.0 for k in ks}, 0
    for start in tqdm(range(0, n_runs, chunk_size), desc=desc, leave=False):
        n_chunk = min(chunk_size, n_runs - start)
        scores = rng.gumbel(size=(n_chunk, len(labels), len(class_prior))).astype(np.float32)
        scores += log_prior[None, None]
        top = np.argpartition(scores, -max_k, axis=2)[:, :, -max_k:]
        top_scores = np.take_along_axis(scores, top, axis=2)
        top = np.take_along_axis(top, np.argsort(-top_scores, axis=2), axis=2)
        for k in ks:
            accuracy = (top[:, :, :k] == labels[None, :, None]).any(axis=2).mean(axis=1)
            sums[k] += float(accuracy.sum())
            sums_squared[k] += float((accuracy ** 2).sum())
        total += n_chunk
    output = {}
    for k in ks:
        mean = sums[k] / total
        variance = max(sums_squared[k] / total - mean**2, 0.0)
        output[k] = float(mean), float(np.sqrt(variance))
    return output

def prior_aware_random_macro_f1(labels, class_prior, n_runs=N_RANDOM_BASELINE, seed=SEED, desc="Macro-F1 random"):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=np.int64)
    class_prior = np.asarray(class_prior, dtype=np.float64)
    class_prior /= class_prior.sum()
    scores = np.empty(n_runs, dtype=np.float32)
    for run in tqdm(range(n_runs), desc=desc, leave=False):
        random_predictions = rng.choice(len(class_prior), size=len(labels), p=class_prior)
        scores[run] = f1_score(labels, random_predictions, average="macro", zero_division=0)
    return float(scores.mean()), float(scores.std(ddof=0))

def indices_by_class(labels):
    labels = np.asarray(labels, dtype=np.int64)
    return {int(label): np.where(labels == label)[0] for label in np.unique(labels)}

def sample_balanced_class_pairs(labels, n_pairs=N_2V2_PAIRS, seed=SEED):
    rng = np.random.default_rng(seed)
    by_class = indices_by_class(labels)
    classes = np.asarray([label for label, indices in by_class.items() if len(indices)], dtype=np.int64)
    if len(classes) < 2:
        raise ValueError("2v2 requires at least two classes")
    first_pos = rng.integers(0, len(classes), size=n_pairs)
    second_pos = (first_pos + rng.integers(1, len(classes), size=n_pairs)) % len(classes)
    first_classes, second_classes = classes[first_pos], classes[second_pos]
    first_idx, second_idx = np.empty(n_pairs, dtype=np.int64), np.empty(n_pairs, dtype=np.int64)
    for label in classes:
        first_mask, second_mask = first_classes == label, second_classes == label
        if first_mask.any():
            first_idx[first_mask] = rng.choice(by_class[int(label)], size=int(first_mask.sum()), replace=True)
        if second_mask.any():
            second_idx[second_mask] = rng.choice(by_class[int(label)], size=int(second_mask.sum()), replace=True)
    return first_idx, second_idx

def probability_2v2(probs, labels, n_pairs=N_2V2_PAIRS, seed=SEED):
    probs, labels = np.asarray(probs, dtype=np.float32), np.asarray(labels, dtype=np.int64)
    first_idx, second_idx = sample_balanced_class_pairs(labels, n_pairs, seed)
    first_labels, second_labels = labels[first_idx], labels[second_idx]
    correct = probs[first_idx, first_labels] + probs[second_idx, second_labels]
    swapped = probs[first_idx, second_labels] + probs[second_idx, first_labels]
    scores = (correct > swapped).astype(np.float32)
    scores += 0.5 * (correct == swapped)
    return float(scores.mean())

eval_items = build_prediction_cache(run_records)
random_stats_by_fold, rows = {}, []
for item in tqdm(eval_items, desc="evaluate folds"):
    fold_id = item["fold_id"]
    labels, probs, ranked = item["y_true"], item["probs"], item["ranked"]
    if fold_id not in random_stats_by_fold:
        train_labels = y_class[item["train_idx"]]
        prior = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
        prior = prior / prior.sum()
        random_stats_by_fold[fold_id] = prior_aware_random_topk(
            labels, prior, seed=derive_seed("topk_random_baseline", fold_id), desc=f"fold {fold_id} Top-k random"
        )
    predictions = np.argmax(probs, axis=1)
    random_macro_mean, random_macro_std = prior_aware_random_macro_f1(
        labels, prior, seed=derive_seed("macro_f1_random_baseline", fold_id),
        desc=f"fold {fold_id} Macro-F1 random"
    )
    row = {
        "fold": fold_id,
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "random_macro_f1_mean": random_macro_mean,
        "random_macro_f1_std": random_macro_std,
        "2v2": probability_2v2(probs, labels, seed=derive_seed("two_v_two", fold_id)),
    }
    for k in TOPK_LIST:
        row[f"top{k}"] = topk_accuracy(labels, ranked, k)
        row[f"random_top{k}_mean"], row[f"random_top{k}_std"] = random_stats_by_fold[fold_id][k]
    rows.append(row)

fold_metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)
metric_columns = ["fold", *[f"top{k}" for k in TOPK_LIST], "macro_f1", "2v2"]
random_columns = [
    "fold", *[column for k in TOPK_LIST for column in (f"random_top{k}_mean", f"random_top{k}_std")],
    "random_macro_f1_mean", "random_macro_f1_std",
]
display(fold_metrics_df[metric_columns])
display(fold_metrics_df[random_columns])

## Cross-Validation Summary and Figures

This final cell aggregates the fold-level results and reports each metric as **mean ± standard deviation across the evaluated folds**. The summary includes Top-k accuracy, Macro-F1, and balanced 2v2 accuracy together with their corresponding random or theoretical chance references.

It also generates publication-oriented PDF figures for the selected subject, including Top-k decoding performance, Macro-F1, and 2v2 accuracy. All plots are saved to the configured output directory for later comparison with the contrastive EEG-language framework.


In [ ]:
def add_bar_labels(axis, bars, means, stds, fmt="{:.3f}", pad=0.012, fontsize=8.5):
    for bar, mean, std in zip(bars, means, stds):
        axis.text(
            bar.get_x() + bar.get_width() / 2, float(mean) + float(std) + pad,
            f"{fmt.format(float(mean))}\n±{fmt.format(float(std))}",
            ha="center", va="bottom", fontsize=fontsize, fontweight="bold",
        )

def random_baseline_mean_std(prefix):
    mean, std = mean_std(fold_metrics_df[f"{prefix}_mean"])
    if len(fold_metrics_df) == 1:
        std = float(fold_metrics_df[f"{prefix}_std"].iloc[0])
    return mean, std

summary_rows = []
for k in TOPK_LIST:
    model_mean, model_std = mean_std(fold_metrics_df[f"top{k}"])
    random_mean, random_std = random_baseline_mean_std(f"random_top{k}")
    summary_rows.append({
        "metric": f"Top-{k}", "mean": model_mean, "std": model_std,
        "reference": "Prior-aware random", "reference_mean": random_mean, "reference_std": random_std,
    })
macro_mean, macro_std = mean_std(fold_metrics_df["macro_f1"])
random_macro_mean, random_macro_std = random_baseline_mean_std("random_macro_f1")
twovtwo_mean, twovtwo_std = mean_std(fold_metrics_df["2v2"])
summary_rows.extend([
    {"metric": "Macro-F1", "mean": macro_mean, "std": macro_std, "reference": "Prior-aware random", "reference_mean": random_macro_mean, "reference_std": random_macro_std},
    {"metric": "2v2", "mean": twovtwo_mean, "std": twovtwo_std, "reference": "Chance", "reference_mean": 0.5, "reference_std": 0.0},
])
summary_df = pd.DataFrame(summary_rows)
summary_display_df = summary_df.copy()
summary_display_df["mean ± SD"] = summary_display_df.apply(lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}", axis=1)
summary_display_df["reference value"] = summary_display_df.apply(
    lambda row: "—" if pd.isna(row["reference_mean"]) else f"{row['reference_mean']:.4f} ± {row['reference_std']:.4f}", axis=1
)
display(summary_display_df[["metric", "mean ± SD", "reference", "reference value"]])

x = np.arange(len(TOPK_LIST))
model_means = np.asarray([mean_std(fold_metrics_df[f"top{k}"])[0] for k in TOPK_LIST])
model_stds = np.asarray([mean_std(fold_metrics_df[f"top{k}"])[1] for k in TOPK_LIST])
random_means = np.asarray([random_baseline_mean_std(f"random_top{k}")[0] for k in TOPK_LIST])
random_stds = np.asarray([random_baseline_mean_std(f"random_top{k}")[1] for k in TOPK_LIST])
fig, axis = plt.subplots(figsize=(10.2, 5.8))
width = 0.36
model_bars = axis.bar(x - width / 2, model_means, width, yerr=model_stds, capsize=4, label="Classifier", color=PAPER_PALETTE["model"], edgecolor="black", linewidth=0.7)
random_bars = axis.bar(x + width / 2, random_means, width, yerr=random_stds, capsize=4, label="Prior-aware random", color=PAPER_PALETTE["random"], edgecolor="black", linewidth=0.7)
add_bar_labels(axis, model_bars, model_means, model_stds, fmt="{:.1%}", pad=0.01, fontsize=7.5)
add_bar_labels(axis, random_bars, random_means, random_stds, fmt="{:.1%}", pad=0.01, fontsize=7.2)
axis.set_xticks(x, [f"Top-{k}" for k in TOPK_LIST])
axis.set(ylabel="Accuracy", title=f"{SUBJECT_ID} softmax Top-k decoding")
axis.set_ylim(0, min(1.0, max(np.max(model_means + model_stds), np.max(random_means + random_stds)) + 0.18))
axis.grid(axis="y", alpha=0.25)
axis.legend(frameon=False)
save_pdf(f"{SUBJECT_ID}_classification_topk")

fig, axis = plt.subplots(figsize=(5.8, 4.8))
macro_means = np.asarray([macro_mean, random_macro_mean])
macro_stds = np.asarray([macro_std, random_macro_std])
bars = axis.bar(
    np.arange(2), macro_means, yerr=macro_stds, capsize=6, width=0.58,
    color=[PAPER_PALETTE["model"], PAPER_PALETTE["random"]], edgecolor="black", linewidth=0.8,
)
add_bar_labels(axis, bars, macro_means, macro_stds)
axis.set_xticks(np.arange(2), ["Classifier", "Prior-aware random"])
axis.set(ylabel="F1 score", title=f"{SUBJECT_ID} classification Macro-F1")
axis.set_ylim(0, min(1.0, np.max(macro_means + macro_stds) + 0.12))
axis.grid(axis="y", alpha=0.25)
save_pdf(f"{SUBJECT_ID}_classification_macro_f1")

fig, axis = plt.subplots(figsize=(4.8, 4.8))
bars = axis.bar([0], [twovtwo_mean], yerr=[twovtwo_std], capsize=6, width=0.55, color=PAPER_PALETTE["model"], edgecolor="black", linewidth=0.8)
add_bar_labels(axis, bars, [twovtwo_mean], [twovtwo_std])
axis.axhline(0.5, color="black", linestyle=":", linewidth=1.5, label="Chance = 0.5")
axis.set_xticks([0], ["2v2"])
axis.set(ylabel="2v2 accuracy", title=f"{SUBJECT_ID} probability-based 2v2")
axis.set_ylim(0.35, min(1.0, max(0.5, twovtwo_mean + twovtwo_std) + 0.14))
axis.grid(axis="y", alpha=0.25)
axis.legend(frameon=False)
save_pdf(f"{SUBJECT_ID}_classification_2v2")

print("Done. PDF plots saved to:", PLOTS_DIR)